# Quick Start for "Residual Learning in Empirical Asset Pricing"


In [1]:
import random
import platform
import subprocess
import numpy as np
import pandas as pd
import torch

from pandas.tseries.offsets import MonthEnd, YearEnd
from zoneinfo import ZoneInfo
from datetime import datetime
from main import parser
from run.dataset import FinancialDataloader
from run.trainer import Trainer
from run.predictor import Predictor
from run.econ import EconEvaluator
from config.resnetp.resnetps4d13 import ResNetpS4D13Config
from utils.cublas_curand import cublas_ver, curand_ver

args = parser.parse_args([
    "--asset_prefix", "quick_start"
])

config = ResNetpS4D13Config(args)


info = {
    'author': 'Dexin PENG',
    'affiliation': 'HKUST(GZ)',
    'python': platform.python_version(),
    'host os': platform.platform(),
    'torch': torch.__version__,
    'cudnn': torch.backends.cudnn.version(),
    'capability': torch.cuda.get_device_capability() if torch.cuda.is_available() else None,
    'cublas': cublas_ver,
    'curand': curand_ver,
    'last_run': datetime.now().astimezone(ZoneInfo("Asia/Shanghai")).strftime('%Y-%m-%d %H:%M:%S %z'),
}

display(pd.DataFrame([info], index=pd.Index(['Residual Learning in Empirical Asset Pricing'])))
if torch.cuda.is_available():
    print('\n'.join(subprocess.check_output(['nvidia-smi']).decode().splitlines()[:4]))


,author,affiliation,python,host os,torch,cudnn,capability,cublas,curand,last_run
Residual Learning in Empirical Asset Pricing,Dexin PENG,HKUST(GZ),3.13.5,Linux-4.18.0-372.9.1.el8.x86_64-x86_64-with-gl...,2.7.1+cu128,90701,"(9, 0)",120803,10309,2026-09-25 16:17:23 +0800


Fri Sep 25 16:17:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.10              Driver Version: 570.86.10      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+


To reproduce the reference environment, use the same combination:

**1. PyTorch 2.7.1+cu128**

**2. cuBLAS 120803**

**3. Compute capability (9, 0)**

Other versions or devices may produce slightly different results.


# I. Data Inspection


## Load

**NOTE**: According to the user manual of the original data, if you use the data `source_data/datashare_with_return.pkl`, cite the paper "Is There a Replication Crisis in Finance?" (2023)

```tex
@article{JensenKellyPedersen2023,
	author = {Jensen, Theis Ingerslev and Kelly, Bryan and Pedersen, Lasse Heje},
	title = {Is There a Replication Crisis in Finance?},
	journal = {The Journal of Finance},
	volume = {78},
	number = {5},
	pages = {2465-2518},
	year = {2023}
}
```

For detailed data instructions, see the source repository: https://github.com/bkelly-lab/ReplicationCrisis


In [2]:
# For those who have access to CRSP, you may download data according to the instructions on https://github.com/bkelly-lab/ReplicationCrisis, and check the `utils/sas7bdat2pickle.py` to generate the corresponding US data `datashare_with_return.pkl`
!python3 ./utils/sas7bdat2pickle.py

price_characteristics = pd.read_pickle('./source_data/datashare_with_return.pkl')
price_characteristics.rename(columns={'ret_exc_lead1m':'RET'}, inplace=True)

Reading source sas file...
Handling source sas file...


In [3]:
# The column 'me' is duplicated
print((price_characteristics.market_equity.dropna() == price_characteristics.me.dropna()).all())
price_characteristics.drop('me', axis=1, inplace=True)

True


In [4]:
price_characteristics.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3744603 entries, 0 to 3744602
Columns: 156 entries, niq_su to RET
dtypes: datetime64[s](1), float64(154), int64(1)
memory usage: 4.4 GB


The 156 columns include `permno`, `DATE`, and `RET`. The remaining 153 columns are JKP firm characteristics.


In [5]:
!ls -lh ./source_data

total 94G
-rw-r--r-- 1 apulis-dev root   29K Sep 24 21:57  F-F_Research_Data_5_Factors_2x3.csv
-rw-r--r-- 1      13334 10034  40K Sep 22 11:28 'Factor Details.xlsx'
-rw-r--r-- 1      13334 10034 1.1M Sep 22 11:30  PredictorData2024.xlsx
-rw-r--r-- 1      13334 10034 3.2K Sep 22 11:30  cluster_labels.csv
-rw-r--r-- 1 apulis-dev root  4.4G Sep 25 16:22  datashare_with_return.pkl
-rw-r--r-- 1 apulis-dev root   49G Sep 25 16:22  t9ftbki1xauaox25.pkl
-rw-r--r-- 1      13334 10034  41G Sep 22 11:28  t9ftbki1xauaox25.sas7bdat


**NOTE**: If you use the data provided here `source_data/PredictorData2024.xlsx`, it's recommended to cite "A Comprehensive Look at The Empirical Performance of Equity Premium Prediction" (2008)

```tex
@article{welch2008ComprehensiveLookEmpirical,
  title   = {A {{Comprehensive Look}} at {{The Empirical Performance}} of {{Equity Premium Prediction}}},
  author  = {Welch, Ivo and Goyal, Amit},
  year    = {2008},
  month   = jul,
  journal = {The Review of Financial Studies},
  volume  = {21},
  number  = {4},
  pages   = {1455--1508},
  issn    = {0893-9454},
  doi     = {10.1093/rfs/hhm014},
  urldate = {2025-06-11}
}
```

In [6]:
# Data source: Amit Goyal’s website
# https://sites.google.com/view/agoyal145
# To get data, search 'Updated data' on Amit Goyal’s website
# A Comprehensive Look at The Empirical Performance of Equity Premium Prediction, Amit Goyal, Ivo Welch, 2008

# NOTE: If you use the data provided here `source_data/PredictorData2024.xlsx`, it's recommended to cite "A Comprehensive Look at The Empirical Performance of Equity Premium Prediction" (2008)
macro_predictors_M = pd.read_excel('source_data/PredictorData2024.xlsx', sheet_name='Monthly')
macro_predictors_Q = pd.read_excel('source_data/PredictorData2024.xlsx', sheet_name='Quarterly')
macro_predictors_A = pd.read_excel('source_data/PredictorData2024.xlsx', sheet_name='Annual')

/usr/local/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


**NOTE**: If you use the data provided here `source_data/F-F_Research_Data_5_Factors_2x3.csv`, it's recommended to cite "A five-factor asset pricing model" (2015)

```tex
@article{fama2015FivefactorAssetPricing,
  title = {A Five-Factor Asset Pricing Model},
  author = {Fama, Eugene F. and French, Kenneth R.},
  year = 2015,
  month = apr,
  journal = {Journal of Financial Economics},
  volume = {116},
  number = {1},
  pages = {1--22},
  issn = {0304405X},
  doi = {10.1016/j.jfineco.2014.10.010},
  urldate = {2025-08-30},
  langid = {english}
}
```

In [7]:
# Data source: Kenneth French's website
# https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html#Research
# To get data, search 'Fama/French 5 Factors (2x3)' on Kenneth French's website and download the `csv` format

# NOTE: If you use the data provided here `source_data/F-F_Research_Data_5_Factors_2x3.csv`, it's recommended to cite "A five-factor asset pricing model" (2015)

ff5 = pd.read_csv('source_data/F-F_Research_Data_5_Factors_2x3.csv', index_col=0)
ff5.index = pd.to_datetime(ff5.index, format='%Y%m') + pd.offsets.MonthEnd(0)

You may notice the discrepancy between the RF in ff5 and the tbl in macro data

The former is the one-month Treasury bill’s monthly return, whereas the latter is the annualized yield (discount basis, TB3MS) on the three-month Treasury bill. They differ in ways like maturity and measurement (return versus yield)


## Date Conversion


To ease computing, use a standard time datatype

E.g. `string` 200001 is converted to a standard time datatype `datetime64[ns]` 2000-01-31 00:00:00 000000000


Using the last day for all different frequencies will make it neutral for all different frequencies

The data source `price_characteristics` picks the last day; to be consistent, pick the last day for all different frequencies

In [8]:
# Time zone is the local time of the dataset, not UTC
macro_predictors_M['DATE'] = pd.to_datetime(macro_predictors_M['yyyymm'], format='%Y%m') + pd.offsets.MonthEnd(0)
macro_predictors_Q['DATE'] = pd.PeriodIndex(
    macro_predictors_Q['yyyyq'].astype(str).str.slice(0, 4) + 'Q' + macro_predictors_Q['yyyyq'].astype(str).str.slice(4),
    freq='Q').to_timestamp() + pd.offsets.QuarterEnd(0)
macro_predictors_A['DATE'] = pd.to_datetime(macro_predictors_A['yyyy'], format='%Y') + pd.offsets.YearEnd(0)

ff5.index = pd.to_datetime(ff5.index, format='%Y%m') + pd.offsets.MonthEnd(0)

Save the ff5 data for later use in `./run/econ.py`

In [9]:
ff5.to_pickle(config.io.clean_ff5_path)

## Index Duplicates Check

In [10]:
price_characteristics.loc[:, ['permno', 'DATE']].duplicated().sum()

np.int64(0)

In [11]:
macro_predictors_M.loc[:, 'DATE'].duplicated().sum()

np.int64(0)

## Indexing & Slicing Dataset

In [12]:
price_characteristics.set_index(['permno', 'DATE'], inplace=True)
price_characteristics.sort_index(inplace=True)
price_characteristics = price_characteristics.loc[(slice(None), slice('1963-01-01', '2023-12-31')), :]
macro_predictors_M.set_index('DATE', inplace=True)
macro_predictors_M.sort_index(inplace=True)

# II. Preprocess

## Missing Value

In [13]:
pd.isna(price_characteristics.RET).sum()

np.int64(0)

In [14]:
from utils.preprocess_handler import PreprocessHandler

preprocess_handler = PreprocessHandler(price_characteristics)
preprocess_handler.fill_missing_value_with_median

# After median filling, the cross-sectionally-all-missing characteristics are still missing.
# Then we fill 0 into the cross-sectionally-all-missing characteristics.

Characteristics filled with median count: 116017179, percentage: 21.07%
Cross-sectionally all missing characteristics: 1568999, percentage: 0.28%


## Split Response Variable

In [15]:
y = price_characteristics.pop('RET')

## Market Equity Backup

Market equity will be used in later econ evaluation. Certain information will be lost if rank normalized 

In [16]:
mvel1 = price_characteristics.market_equity
mvel1.to_pickle(config.io.clean_mvel1_path)
mvel1 = pd.read_pickle(config.io.clean_mvel1_path)

## Rank Normalization

Cross-sectionally rank all stock characteristics and map these ranks into the [-1,1] interval

In [17]:
price_characteristics[:] = price_characteristics.groupby('DATE').transform(lambda s: s.rank(pct=True) * 2 - 1)

In [18]:
# After median filling, the cross-sectionally-all-missing characteristics are still missing.
# Then we fill 0 into the cross-sectionally-all-missing characteristics.
# After rank normalization, 0 is cross-sectionally the median for every characteristic; then filling with 0 is another kind of filling with the median
price_characteristics.fillna(0, inplace=True)

## Merge Macro Data

In [19]:
macro_predictors_M.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1848 entries, 1871-01-31 to 2024-12-31
Columns: 57 entries, yyyymm to disag
dtypes: float64(56), int64(1)
memory usage: 837.4 KB


In [20]:
# Use dividend-price ratio (d/p), earnings-price ratio (e/p), book-to-market ratio (b/m), net equity expansion (ntis), Treasury-bill rate (tbl), term spread (tms), default spread (dfy), and stock variance (svar) according to Gu et al. 2020 page 2248, Paragraph 3 of Section 2.1
selected_macro = macro_predictors_M[['d/p', 'e/p', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']].loc['19630101':'20231231']
pd.isna(selected_macro).sum().sum()

np.int64(0)

No missing data after 1927


In [21]:
X = price_characteristics.join(selected_macro)

In [22]:
X.to_pickle(config.io.clean_X_path)
y.to_pickle(config.io.clean_y_path)


In [23]:
!ls -lh tmp


total 23G
-rw-r--r-- 1 apulis-dev root  18G Sep 25 01:13 ResnetAssetPricing_pretrained_checkpoints_20260925_005919.zip
-rw-r--r-- 1 apulis-dev root 4.4G Sep 25 16:24 clean_X.pkl
-rw-r--r-- 1 apulis-dev root  42K Sep 25 16:22 clean_ff5.pkl
-rw-r--r-- 1 apulis-dev root  55M Sep 25 16:23 clean_mvel1.pkl
-rw-r--r-- 1 apulis-dev root  55M Sep 25 16:24 clean_y.pkl


## Train Test Split

The market is dynamic and a rolling split is practical. Every time we receive new data, re-train the model with the latest adjusted dataset

In [24]:
# This research dataset stops in (2023,12,31)

# For example

X_train, X_test, y_train, y_test = config.data.split_data(X, y, now=datetime.now(), freq='YE')
print(f'\nAnnual rebalance, for example, now is {datetime.now().strftime("%Y-%m-%d")}, dataset should be')
print(f'X_train: {X_train.index.get_level_values("DATE").min().date()} to {X_train.index.get_level_values("DATE").max().date()}')
print(f'X_test: {config.data.test_start.strftime("%Y-%m-%d")} to {config.data.test_end.strftime("%Y-%m-%d")}')

X_train, X_test, y_train, y_test = config.data.split_data(X, y, now=datetime.now() + YearEnd(0), freq='YE')
print(f'\nAnnual rebalance, for example, when it comes to the end of this year, {(datetime.now() + YearEnd(0)).strftime("%Y-%m-%d")}, at that time, dataset should be')
print(f'The most near future: {(datetime.now() + YearEnd(0)).strftime("%Y-%m-%d")}')
print(f'X_train: {X_train.index.get_level_values("DATE").min().date()} to {X_train.index.get_level_values("DATE").max().date()}')
print(f'X_test: {config.data.test_start.strftime("%Y-%m-%d")} to {config.data.test_end.strftime("%Y-%m-%d")}')



Annual rebalance, for example, now is 2026-09-25, dataset should be
X_train: 1963-01-31 to 2023-11-30
X_test: 2025-01-31 to 2025-12-31

Annual rebalance, for example, when it comes to the end of this year, 2026-12-31, at that time, dataset should be
The most near future: 2026-12-31
X_train: 1963-01-31 to 2023-11-30
X_test: 2026-01-31 to 2026-12-31


## Comprehensive Data Check

If you use other sources of data or modify some settings, it's recommended to use this cell to check the integrity of data; this will help you pre-discover a lot of bugs in later steps

If you are using the same source data and config settings as this research, you may want to skip this cell to save some time


In [ ]:
# # Items to be checked
# source data matches the number of explanatory variables for this research
# shape consistency of variables across training and test sets
# shape consistency between explanatory and response variables
# index consistency between explanatory and response variables
# there are data points in each split set
# proper data type
# no missing values
# no infinite values
# no duplicate datetime-stkcd observations
# no non-macro values greater than 1 or less than -1

for now in pd.date_range(start=config.data.time_loop_start, end=config.data.time_loop_end, freq=config.data.time_loop_freq):
    X_train, X_test, y_train, y_test = config.data.split_data(X, y, now=now, freq=config.data.time_loop_freq)
    PreprocessHandler.comprehensive_data_check(X_train, X_test, y_train, y_test, config, now=now)


Checking: Now: 1987-12-31, Train: 1963-01-31 to 1986-12-31, Test: 1987-01-31 to 1987-12-31
Pass
Checking: Now: 1988-12-31, Train: 1963-01-31 to 1987-12-31, Test: 1988-01-31 to 1988-12-31
Pass
Checking: Now: 1989-12-31, Train: 1963-01-31 to 1988-12-31, Test: 1989-01-31 to 1989-12-31
Pass
Checking: Now: 1990-12-31, Train: 1963-01-31 to 1989-12-31, Test: 1990-01-31 to 1990-12-31
Pass
Checking: Now: 1991-12-31, Train: 1963-01-31 to 1990-12-31, Test: 1991-01-31 to 1991-12-31
Pass
Checking: Now: 1992-12-31, Train: 1963-01-31 to 1991-12-31, Test: 1992-01-31 to 1992-12-31
Pass
Checking: Now: 1993-12-31, Train: 1963-01-31 to 1992-12-31, Test: 1993-01-31 to 1993-12-31
Pass
Checking: Now: 1994-12-31, Train: 1963-01-31 to 1993-12-31, Test: 1994-01-31 to 1994-12-31
Pass
Checking: Now: 1995-12-31, Train: 1963-01-31 to 1994-12-31, Test: 1995-01-31 to 1995-12-31
Pass
Checking: Now: 1996-12-31, Train: 1963-01-31 to 1995-12-31, Test: 1996-01-31 to 1996-12-31
Pass
Checking: Now: 1997-12-31, Train: 1963-0

## E.g. To `torch.Tensor`

In [ ]:
from run.dataset import FinancialDataloader

now = datetime(2023, 12, 31)

# Test data will not be involved in training under any circumstances
data_loader = FinancialDataloader(config, now, X, y)

print(f'For example, now={now.strftime("%Y-%m-%d")}')
print(f'X_train_tensor.shape={data_loader.X_train_tensor.shape}, type={type(data_loader.X_train_tensor)}')
print(f'X_test_tensor.shape={data_loader.X_test_tensor.shape}, type={type(data_loader.X_test_tensor)}')
print(f'y_train_tensor.shape={data_loader.y_train_tensor.shape}, type={type(data_loader.y_train_tensor)}')
print(f'y_test_tensor.shape={data_loader.y_test_tensor.shape}, type={type(data_loader.y_test_tensor)}')


For example, now=2023-12-31
X_train_tensor.shape=torch.Size([3519102, 161]), type=<class 'torch.Tensor'>
X_test_tensor.shape=torch.Size([56380, 161]), type=<class 'torch.Tensor'>
y_train_tensor.shape=torch.Size([3519102, 1]), type=<class 'torch.Tensor'>
y_test_tensor.shape=torch.Size([56380, 1]), type=<class 'torch.Tensor'>


# III. Pre-split Data

This part is optional; you can skip this part, but it's recommended to run it


Initialize the pre-split dataset to reduce the time cost of dynamic data splitting, loading and copying

If you set `--init_pre_split_dataset`, the dataset will be pre-split and saved to the `tmp` directory

Later you can specify `--use_pre_split_dataset` in command line arguments


If this part is skipped, you may experience a bottleneck on CPU, memory or disk


In [27]:
config.data.init_pre_split_dataset = True

for now in pd.date_range(start=config.data.time_loop_start, end=config.data.time_loop_end, freq=config.data.time_loop_freq):
    dataloader = FinancialDataloader(config, now, X, y)
    del dataloader
    torch.cuda.empty_cache()


2026-09-25 16:27:29,670 - runner - INFO - Pre split dataset for now=1987-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:31,349 - runner - INFO - Pre split dataset for now=1988-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:33,064 - runner - INFO - Pre split dataset for now=1989-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:34,903 - runner - INFO - Pre split dataset for now=1990-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:36,844 - runner - INFO - Pre split dataset for now=1991-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:38,878 - runner - INFO - Pre split dataset for now=1992-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:40,995 - runner - INFO - Pre split dataset for now=1993-12-31 saved to /data/user/dpeng965/ResnetAssetPricing/tmp
2026-09-25 16:27:43,318 - runner - INFO - Pre split dataset for now=1994-12-31 save

In [28]:
# Revert this setting after pre-splitting data for later coding
config.data.init_pre_split_dataset = False


In [29]:
!du -h --total ./tmp/*

754M	./tmp/19871231
802M	./tmp/19881231
849M	./tmp/19891231
894M	./tmp/19901231
939M	./tmp/19911231
985M	./tmp/19921231
1.1G	./tmp/19931231
1.1G	./tmp/19941231
1.2G	./tmp/19951231
1.2G	./tmp/19961231
1.3G	./tmp/19971231
1.3G	./tmp/19981231
1.4G	./tmp/19991231
1.4G	./tmp/20001231
1.5G	./tmp/20011231
1.5G	./tmp/20021231
1.6G	./tmp/20031231
1.6G	./tmp/20041231
1.6G	./tmp/20051231
1.7G	./tmp/20061231
1.7G	./tmp/20071231
1.8G	./tmp/20081231
1.8G	./tmp/20091231
1.8G	./tmp/20101231
1.8G	./tmp/20111231
1.9G	./tmp/20121231
1.9G	./tmp/20131231
1.9G	./tmp/20141231
2.0G	./tmp/20151231
2.0G	./tmp/20161231
2.0G	./tmp/20171231
2.1G	./tmp/20181231
2.1G	./tmp/20191231
2.1G	./tmp/20201231
2.2G	./tmp/20211231
2.2G	./tmp/20221231
2.2G	./tmp/20231231
18G	./tmp/ResnetAssetPricing_pretrained_checkpoints_20260925_005919.zip
4.4G	./tmp/clean_X.pkl
44K	./tmp/clean_ff5.pkl
55M	./tmp/clean_mvel1.pkl
55M	./tmp/clean_y.pkl
79G	total


# IV. Train

**If you have pre-split the dataset, run the ``import`` cell and then continue here.**

**Check `./tmp` directory to see whether you have successfully pre-split the dataset**

**You may restart the Jupyter kernel now to clear the cache of previous cells**


It's recommended to train in a terminal instead of in a Jupyter notebook

1. terminal execution avoids potential deadlocks or process‑management quirks that sometimes occur when using multiprocessing in Notebook kernels
2. streaming large amounts of training logs into Notebook cells will bloat the `.ipynb` file and slow down IDE loading and parsing


The result is stochastic; use the 10 random seeds 0, 1, ..., 9 to train the model and pool all of these models' predictions by averaging them as the final prediction

The number is one-to-one mapped to the random seed

In [30]:
# It's recommended to copy it (without the starting ! sign) to a terminal and run there

# If you didn't run Part III. Pre-split Data, delete the flag `--use_pre_split_dataset`

# To make it run once and then succeed, it's safe to have at least 40GB GPU memory; otherwise you may experience a GPU out of memory error
!for i in {0..9}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait

# # In case of a GPU out of memory error, you can run fewer ensembles in multi-process at one time
# !for i in {0..1}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait
# !for i in {2..3}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait
# !for i in {4..5}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait
# !for i in {6..7}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait
# !for i in {8..9}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait

# # Because the results are stochastic, in case you want to run another set of random seeds, e.g. change from seeds 0-9 to 10-19
# # Change self.model.start_ensemble = 0 to 10, self.model.max_ensemble = 9 to 19 in `config/base.py`, or here
# config.model.start_ensemble = 10
# config.model.max_ensemble = 19
# !for i in {10..19}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset & done; wait

# # You may spawn different ensembles on different cuda devices to save time
# !for i in {0..1}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset --device cuda:0 & done; \
# for i in {2..3}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset --device cuda:1 & done; \
# for i in {4..5}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset --device cuda:2 & done; \
# for i in {6..7}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset --device cuda:3 & done; \
# for i in {8..9}; do python3 main.py --ensemble "$i" --train --config_cls ResNetpS4D13Config --epochs 100 --batch_size 10000 --early_stopping_patience 5 --asset_prefix quick_start --l2 --weight_decay 1e-4 --initial_lr 1e-4 --use_pre_split_dataset --device cuda:4 & done; wait


job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%14-12-31 ep...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...
job                resnetps4... done    [#############] 100.0%23-12-31 fi...


# V. Predict

It's recommended to predict in a terminal instead of in a Jupyter notebook

1. terminal execution avoids potential deadlocks or process‑management quirks that sometimes occur when using multiprocessing in Notebook kernels
2. streaming large amounts of training logs into Notebook cells will bloat the `.ipynb` file and slow down IDE loading and parsing


**Before predicting for the specified ensemble, make sure you have the pre-trained checkpoints of the same ensemble's model**

In [31]:
# It's recommended to copy it (without the starting ! sign) to a terminal and run there

# If you didn't run Part III. Pre-split Data, delete the flag `--use_pre_split_dataset`
# Use of the pre-split dataset is to recover the datetime-permno index of the test set, which is lost in tensor format

# The GPU memory requirement is much less than that for training
!for i in {0..9}; do python3 main.py --ensemble "$i" --predict --config_cls ResNetpS4D13Config --asset_prefix quick_start --use_pre_split_dataset & done; wait

# # Because the results are stochastic, in case you want to run another set of random seeds, e.g. change from seeds 0-9 to 10-19
# # Change self.model.start_ensemble = 0 to 10, self.model.max_ensemble = 9 to 19 in `config/base.py`, or here
# config.model.start_ensemble = 10
# config.model.max_ensemble = 19
# !for i in {10..19}; do python3 main.py --ensemble "$i" --predict --config_cls ResNetpS4D13Config --asset_prefix quick_start --use_pre_split_dataset --start_ensemble 10 --max_ensemble 19 & done; wait


job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%
job                resnetps4... done    [#############] 100.0%
job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%23-12-31 sa...
job                resnetps4... done    [#############] 100.0%23-12-31 sa...


# VI. Economic Evaluation


**Average the forecasts of all ten ensemble members for economic evaluation. Run this section after training and prediction have completed for every ensemble.**


In [32]:
args.econ = True
config.econ.econ_mkt_cap = ['ALL', 'TOP90%', 'TOP80%']


Detailed results are written to `asset/quick_start/econ`


In [33]:
econ = EconEvaluator(config)

[econ] quick_start                        ALL          SR  vw +2.138
[econ] quick_start                        TOP90%       SR  vw +1.995
[econ] quick_start                        TOP80%       SR  vw +1.853


## VII. All Experiments

Congratulations! You've successfully run the 13-layer ResNet+.

Use `./run.sh` to interactively run remaining experiments.

Check `./utils/paper_source/exhibits.ipynb` for figures and tables used in the paper.